# HarmonicMoE: Spatial + Transformer + Fourier на двух GPU

Более сложная финальная архитектура использует общий plane-wise encoder и три bottleneck-эксперта:

1. spatial residual expert для локальной структуры;
2. cross-plane Transformer для глобального конфликта четырёх глубин;
3. Fourier expert для спектрального распределения и спекл-шума.

Обучаемый gate выбирает смесь экспертов для каждой четвёрки. `DataParallel` оборачивает всю систему —
backbone, Френель и loss — поэтому FFT также считается на обеих GPU. Перед обучением изображения один раз
рендерятся в float16 cache: высокая CPU-нагрузка остаётся только на стадии построения кэша.

In [ ]:
from __future__ import annotations

import csv
import json
import math
import os
import random
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont, ImageOps
from torch import nn
from torch.utils.data import DataLoader, Dataset

try:
    from pytorch_msssim import ssim
except ImportError:
    print('Installing pytorch-msssim...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pytorch-msssim'])
    from pytorch_msssim import ssim

KAGGLE_DATASET_DIR = Path('/kaggle/input/ai-phonetics-data-v2')
OUTPUT_DIR = Path('/kaggle/working/moe_dataparallel_v2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Set this to last.pt from an attached Kaggle Dataset to resume across sessions.
RESUME_CHECKPOINT = None
BASE_CHECKPOINT = None  # optional best/last checkpoint from SharedDepthUNet

IMAGE_SIZE = 128
WAVELENGTH = 532e-9
PIXEL_PITCH = 10e-6
ZS = (24.1e-3, 25.3e-3, 26.5e-3, 27.7e-3)

TRAIN_SEED = 42
MAX_EPOCHS = 30
MIN_EPOCHS = 12
EARLY_STOP_PATIENCE = 6
BATCH_SIZE = 32  # 16 samples per GPU for 2 x T4; try 64 if memory allows
LR = 3e-4
MIN_LR = 1e-5
WEIGHT_DECAY = 1e-4
SSIM_WEIGHT = 1.5
NUM_WORKERS = min(2, os.cpu_count() or 1)
LOG_EVERY = 50
CACHE_WORKERS = min(4, os.cpu_count() or 1)
CACHE_DIR = Path('/kaggle/working/render_cache_v2')
GATE_BALANCE_WEIGHT = 0.01

def find_root(path: Path) -> Path:
    if (path / 'dataset_metadata.json').is_file():
        return path
    matches = list(path.rglob('dataset_metadata.json'))
    if len(matches) != 1:
        raise FileNotFoundError(f'Cannot find exactly one dataset_metadata.json below {path}')
    return matches[0].parent

DATA_ROOT = find_root(KAGGLE_DATASET_DIR)
metadata = json.loads((DATA_ROOT / 'dataset_metadata.json').read_text())
if metadata.get('schema_version') != 2:
    raise RuntimeError(f'Expected dataset schema 2, got {metadata.get("schema_version")}')
BSDS_ROOT = DATA_ROOT / 'BSDS500' / 'data'

CONFIG = {
    'architecture': 'HarmonicMoEUNet',
    'dataset_schema': 2,
    'train_seed': TRAIN_SEED,
    'max_epochs': MAX_EPOCHS,
    'min_epochs': MIN_EPOCHS,
    'early_stop_patience': EARLY_STOP_PATIENCE,
    'batch_size': BATCH_SIZE,
    'lr': LR,
    'min_lr': MIN_LR,
    'weight_decay': WEIGHT_DECAY,
    'ssim_weight': SSIM_WEIGHT,
        'gate_balance_weight': GATE_BALANCE_WEIGHT,
    'image_size': IMAGE_SIZE,
    'wavelength': WAVELENGTH,
    'pixel_pitch': PIXEL_PITCH,
    'zs': ZS,
}
(OUTPUT_DIR / 'config.json').write_text(json.dumps(CONFIG, indent=2), encoding='utf-8')
print('DATA_ROOT:', DATA_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'disabled')

In [ ]:
RESAMPLE = Image.Resampling.LANCZOS

def normalize_image(array: np.ndarray) -> torch.Tensor:
    values = torch.from_numpy(np.asarray(array, dtype=np.float32) / 255.0)
    maximum = values.amax()
    if maximum <= 1e-8:
        raise ValueError('All-black image cannot be normalized')
    return values / maximum

def font_for_size(size: int):
    try:
        return ImageFont.truetype('DejaVuSans.ttf', size=size)
    except OSError:
        return ImageFont.load_default()

def draw_primitive(mask: Image.Image, rng: np.random.Generator, remaining_fraction: float) -> None:
    draw = ImageDraw.Draw(mask)
    target_area = max(12.0, remaining_fraction * IMAGE_SIZE * IMAGE_SIZE * float(rng.uniform(0.25, 0.65)))
    kind = str(rng.choice(['ellipse', 'rectangle', 'line', 'ring', 'glyph'], p=[0.28, 0.28, 0.16, 0.14, 0.14]))
    aspect = float(np.exp(rng.uniform(np.log(0.45), np.log(2.2))))
    width = int(np.clip(np.sqrt(target_area * aspect), 3, IMAGE_SIZE - 4))
    height = int(np.clip(np.sqrt(target_area / aspect), 3, IMAGE_SIZE - 4))
    x0 = int(rng.integers(2, max(3, IMAGE_SIZE - width - 1)))
    y0 = int(rng.integers(2, max(3, IMAGE_SIZE - height - 1)))
    x1, y1 = min(IMAGE_SIZE - 2, x0 + width), min(IMAGE_SIZE - 2, y0 + height)

    if kind == 'ellipse':
        draw.ellipse((x0, y0, x1, y1), fill=255)
    elif kind == 'rectangle':
        draw.rectangle((x0, y0, x1, y1), fill=255)
    elif kind == 'line':
        line_width = max(1, int(np.clip(target_area / max(width, height), 1, 10)))
        draw.line((x0, y0, x1, y1), fill=255, width=line_width)
    elif kind == 'ring':
        ring_width = max(1, int(np.clip(min(width, height) * 0.12, 1, 9)))
        draw.ellipse((x0, y0, x1, y1), outline=255, width=ring_width)
    else:
        char = '0123456789ABCDEFGHJKLMNPQRSTUVWXYZ'[int(rng.integers(0, 34))]
        font = font_for_size(max(10, min(80, height)))
        glyph = Image.new('L', (max(16, width * 2), max(16, height * 2)), 0)
        gd = ImageDraw.Draw(glyph)
        box = gd.textbbox((0, 0), char, font=font)
        gd.text((-box[0], -box[1]), char, fill=255, font=font)
        box = glyph.getbbox()
        if box is not None:
            glyph = glyph.crop(box).resize((max(3, width), max(3, height)), RESAMPLE)
            mask.paste(glyph, (x0, y0), glyph)

def make_mask(seed: int, target_coverage: float) -> tuple[Image.Image, float]:
    best_mask, best_coverage, best_error = None, 0.0, float('inf')
    for attempt in range(8):
        rng = np.random.default_rng(seed + 1009 * attempt)
        mask = Image.new('L', (IMAGE_SIZE, IMAGE_SIZE), 0)
        for _ in range(18):
            coverage = float((np.asarray(mask) > 16).mean())
            error = abs(coverage - target_coverage)
            if error < best_error:
                best_mask, best_coverage, best_error = mask.copy(), coverage, error
            if coverage >= target_coverage * 0.96:
                break
            draw_primitive(mask, rng, max(target_coverage - coverage, 0.002))
        coverage = float((np.asarray(mask) > 16).mean())
        error = abs(coverage - target_coverage)
        if error < best_error:
            best_mask, best_coverage, best_error = mask.copy(), coverage, error
        if target_coverage * 0.75 <= coverage <= target_coverage * 1.25:
            return mask, coverage
    return best_mask, best_coverage

def render_synthetic(item: dict, return_coverage: bool = False):
    seed = int(item['seed'])
    rng = np.random.default_rng(seed)
    target = float(item['target_coverage'])
    mask, coverage = make_mask(seed, target)
    mode = item['background_mode']
    if mode == 'dark':
        background = int(rng.integers(0, 19)); foreground = int(rng.integers(max(80, background + 65), 256))
    elif mode == 'light':
        background = int(rng.integers(175, 246)); foreground = int(rng.integers(0, max(1, background - 65)))
    else:
        background = int(rng.integers(25, 101))
        if rng.random() < 0.7:
            foreground = int(rng.integers(min(255, background + 65), 256))
        else:
            foreground = int(rng.integers(0, max(1, background - 20)))
    mask_values = np.asarray(mask, dtype=np.float32) / 255.0
    image = background + mask_values * (foreground - background)
    image = np.clip(np.rint(image), 0, 255).astype(np.uint8)
    tensor = normalize_image(image)
    return (tensor, coverage) if return_coverage else tensor

def load_natural(item: dict, root: Path = BSDS_ROOT) -> torch.Tensor:
    path = root / item['path']
    with Image.open(path) as source:
        image = source.convert('L')
        if item['transform'] == 'full':
            fill = int(np.asarray(image, dtype=np.uint8).mean())
            image = ImageOps.pad(image, (IMAGE_SIZE, IMAGE_SIZE), method=RESAMPLE, color=fill, centering=(0.5, 0.5))
        else:
            crop = item['crop']; size = int(crop['size']); x, y = int(crop['x']), int(crop['y'])
            image = image.crop((x, y, x + size, y + size)).resize((IMAGE_SIZE, IMAGE_SIZE), RESAMPLE)
        if item.get('hflip', False):
            image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        angle = int(item.get('rotation', 0))
        if angle:
            fill = int(np.asarray(image, dtype=np.uint8).mean())
            image = image.rotate(angle, resample=Image.Resampling.BILINEAR, fillcolor=fill)
        return normalize_image(np.asarray(image, dtype=np.uint8))

def load_item(item: dict) -> torch.Tensor:
    return render_synthetic(item) if item['source'] == 'synthetic' else load_natural(item)

class QuadDataset(Dataset):
    def __init__(self, manifest: Path):
        self.records = [json.loads(line) for line in Path(manifest).read_text(encoding='utf-8').splitlines() if line]
    def __len__(self):
        return len(self.records)
    def __getitem__(self, index):
        quad = torch.stack([load_item(item) for item in self.records[index]['items']]).to(torch.float32)
        if quad.shape != (4, IMAGE_SIZE, IMAGE_SIZE):
            raise RuntimeError(f'Invalid quad shape: {tuple(quad.shape)}')
        return quad

class QuadSourceDataset(QuadDataset):
    def __getitem__(self, index):
        record = self.records[index]
        quad = torch.stack([load_item(item) for item in record['items']]).to(torch.float32)
        sources = torch.tensor([1 if item['source'] == 'bsds' else 0 for item in record['items']], dtype=torch.long)
        return quad, sources

In [ ]:
class DoubleConv(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.SiLU(),
                         nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.SiLU())

class UpBlock(nn.Module):
    def __init__(self, cin, skip, cout):
        super().__init__(); self.reduce = nn.Conv2d(cin, cout, 1); self.block = DoubleConv(cout + skip, cout)
    def forward(self, x, skip):
        x = nn.functional.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        return self.block(torch.cat((self.reduce(x), skip), dim=1))

class BaselineUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.e0, self.e1, self.e2, self.e3 = DoubleConv(4, 32), DoubleConv(32, 64), DoubleConv(64, 128), DoubleConv(128, 256)
        self.pool, self.b = nn.MaxPool2d(2), DoubleConv(256, 512)
        self.d3, self.d2, self.d1, self.d0 = UpBlock(512, 256, 256), UpBlock(256, 128, 128), UpBlock(128, 64, 64), UpBlock(64, 32, 32)
        self.head = nn.Conv2d(32, 1, 1)
    def forward(self, x):
        e0 = self.e0(x); e1 = self.e1(self.pool(e0)); e2 = self.e2(self.pool(e1)); e3 = self.e3(self.pool(e2))
        x = self.b(self.pool(e3)); x = self.d3(x, e3); x = self.d2(x, e2); x = self.d1(x, e1); x = self.d0(x, e0)
        return self.head(x)

class DepthEmbedding(nn.Module):
    def __init__(self, channels):
        super().__init__(); hidden = max(16, channels // 2)
        self.net = nn.Sequential(nn.Linear(1, hidden), nn.SiLU(), nn.Linear(hidden, channels))
        nn.init.zeros_(self.net[-1].weight); nn.init.zeros_(self.net[-1].bias)
    def forward(self, depths):
        return self.net(depths[:, None])[:, :, None, None]

class PlaneFusion(nn.Sequential):
    def __init__(self, channels):
        super().__init__(nn.Conv2d(4 * channels, channels, 1, bias=False), nn.GroupNorm(8, channels), nn.SiLU())

class SpatialExpert(nn.Module):
    def __init__(self, channels=512):
        super().__init__()
        hidden = channels // 2
        self.net = nn.Sequential(
            nn.Conv2d(channels, hidden, 1, bias=False), nn.GroupNorm(8, hidden), nn.SiLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1, groups=hidden, bias=False),
            nn.GroupNorm(8, hidden), nn.SiLU(), nn.Conv2d(hidden, channels, 1, bias=False),
        )
    def forward(self, x):
        return x + self.net(x)

class CrossPlaneTransformerExpert(nn.Module):
    def __init__(self, channels=512, planes=4, height=8, width=8, dim=256):
        super().__init__()
        self.planes, self.height, self.width = planes, height, width
        self.input_projection = nn.Linear(channels, dim)
        self.position = nn.Parameter(torch.zeros(1, planes * height * width, dim))
        layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=8, dim_feedforward=3 * dim, dropout=0.1,
            activation='gelu', batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=2)
        self.output_projection = nn.Linear(dim, channels)
        self.fusion = PlaneFusion(channels)
        nn.init.trunc_normal_(self.position, std=0.02)

    def forward(self, plane_features):
        batch, planes, channels, height, width = plane_features.shape
        if (planes, height, width) != (self.planes, self.height, self.width):
            raise RuntimeError(f'Unexpected bottleneck shape: {plane_features.shape}')
        tokens = plane_features.permute(0, 1, 3, 4, 2).reshape(batch, planes * height * width, channels)
        tokens = self.input_projection(tokens) + self.position
        tokens = self.output_projection(self.transformer(tokens))
        maps = tokens.reshape(batch, planes, height, width, channels).permute(0, 1, 4, 2, 3)
        return self.fusion(maps.reshape(batch, planes * channels, height, width))

class FourierExpert(nn.Module):
    def __init__(self, channels=512, planes=4, spectral_channels=128, height=8, width=8):
        super().__init__()
        self.planes = planes
        self.reduce = nn.Conv2d(channels, spectral_channels, 1, bias=False)
        self.weight_real = nn.Parameter(torch.ones(1, spectral_channels, height, width // 2 + 1))
        self.weight_imag = nn.Parameter(torch.zeros(1, spectral_channels, height, width // 2 + 1))
        self.expand = nn.Sequential(
            nn.Conv2d(spectral_channels, channels, 1, bias=False), nn.GroupNorm(8, channels), nn.SiLU(),
        )
        self.fusion = PlaneFusion(channels)

    def forward(self, plane_features):
        batch, planes, channels, height, width = plane_features.shape
        values = self.reduce(plane_features.reshape(batch * planes, channels, height, width))
        spectrum = torch.fft.rfft2(values.float(), norm='ortho')
        spectral_weight = torch.complex(self.weight_real, self.weight_imag)
        values = torch.fft.irfft2(spectrum * spectral_weight, s=(height, width), norm='ortho')
        values = self.expand(values.to(plane_features.dtype))
        values = values.reshape(batch, planes, channels, height, width)
        return self.fusion(values.reshape(batch, planes * channels, height, width))

class HarmonicMoEUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.planes = 4
        channels = (32, 64, 128, 256, 512)
        self.encoders = nn.ModuleList([
            DoubleConv(1, 32), DoubleConv(32, 64), DoubleConv(64, 128),
            DoubleConv(128, 256), DoubleConv(256, 512),
        ])
        self.depth_embeddings = nn.ModuleList([DepthEmbedding(c) for c in channels])
        self.fusions = nn.ModuleList([PlaneFusion(c) for c in channels])
        self.pool = nn.MaxPool2d(2)
        self.spatial_expert = SpatialExpert(512)
        self.transformer_expert = CrossPlaneTransformerExpert(512)
        self.fourier_expert = FourierExpert(512)
        self.gate = nn.Sequential(nn.Linear(16 + 3 * 512, 256), nn.SiLU(), nn.Linear(256, 3))
        self.d3, self.d2 = UpBlock(512, 256, 256), UpBlock(256, 128, 128)
        self.d1, self.d0 = UpBlock(128, 64, 64), UpBlock(64, 32, 32)
        self.head = nn.Conv2d(32, 1, 1)
        z = torch.tensor(ZS, dtype=torch.float32) * 1e3
        self.register_buffer('depths', (z - z.mean()) / (z.max() - z.min()))

    @staticmethod
    def image_statistics(x):
        image = (x + 1) * 0.5
        mean = image.mean(dim=(-2, -1))
        std = image.std(dim=(-2, -1))
        dark = (image < 0.05).to(image.dtype).mean(dim=(-2, -1))
        dx = (image[..., :, 1:] - image[..., :, :-1]).abs().mean(dim=(-2, -1))
        dy = (image[..., 1:, :] - image[..., :-1, :]).abs().mean(dim=(-2, -1))
        edge = 0.5 * (dx + dy)
        return torch.stack((mean, std, dark, edge), dim=-1).flatten(1)

    def forward(self, x):
        batch, planes, height, width = x.shape
        features = x.reshape(batch * planes, 1, height, width)
        depths = self.depths.repeat(batch)
        skips = []
        plane_bottleneck = None
        for level, (encoder, embedding, fusion) in enumerate(zip(self.encoders, self.depth_embeddings, self.fusions)):
            features = encoder(features) + embedding(depths).to(features.dtype)
            _, channels, h, w = features.shape
            planes_view = features.reshape(batch, planes, channels, h, w)
            if level < 4:
                skips.append(fusion(planes_view.reshape(batch, planes * channels, h, w)))
                features = self.pool(features)
            else:
                plane_bottleneck = planes_view

        plain_fused = self.fusions[4](plane_bottleneck.reshape(batch, planes * 512, 8, 8))
        expert_outputs = torch.stack((
            self.spatial_expert(plain_fused),
            self.transformer_expert(plane_bottleneck),
            self.fourier_expert(plane_bottleneck),
        ), dim=1)
        pooled = expert_outputs.mean(dim=(-2, -1)).flatten(1)
        gates = torch.softmax(self.gate(torch.cat((self.image_statistics(x), pooled), dim=1)), dim=1)
        fused = (expert_outputs * gates[:, :, None, None, None]).sum(dim=1)
        e0, e1, e2, e3 = skips
        fused = self.d3(fused, e3); fused = self.d2(fused, e2)
        fused = self.d1(fused, e1); fused = self.d0(fused, e0)
        return self.head(fused), gates

class FresnelLayer(nn.Module):
    def __init__(self, device):
        super().__init__(); c = (torch.arange(IMAGE_SIZE, dtype=torch.float64) - IMAGE_SIZE / 2) * PIXEL_PITCH
        u, v = torch.meshgrid(c, c, indexing='ij')
        kernels = [torch.exp(1j * math.pi * (u.square() + v.square()) / (WAVELENGTH * z)).to(torch.complex64) for z in ZS]
        self.register_buffer('kernels', torch.stack(kernels).to(device), persistent=False)
    def forward(self, phase):
        field = torch.polar(torch.ones_like(phase, dtype=torch.float32), phase.float()); outputs = []
        for kernel in self.kernels:
            value = torch.fft.fftshift(torch.fft.fft2(torch.fft.ifftshift(field * kernel, dim=(-2, -1))), dim=(-2, -1)).abs()
            outputs.append(value)
        amplitude = torch.cat(outputs, dim=1)
        return amplitude / (amplitude.amax(dim=(-2, -1), keepdim=True) + 1e-8)

def pearson_cc(a, b, eps=1e-8):
    a = a - a.mean(dim=(-2, -1), keepdim=True); b = b - b.mean(dim=(-2, -1), keepdim=True)
    return (a * b).sum(dim=(-2, -1)) / (torch.sqrt(a.square().sum(dim=(-2, -1)) * b.square().sum(dim=(-2, -1))) + eps)

def doe_metrics(target, reconstruction):
    cc = pearson_cc(target, reconstruction)
    sv = torch.stack([
        ssim(target[:, m:m+1], reconstruction[:, m:m+1], data_range=1.0, size_average=False)
        for m in range(4)
    ], dim=1)
    return cc, sv

class EndToEndDOE(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = HarmonicMoEUNet()
        self.physics = FresnelLayer(torch.device('cuda:0'))

    def forward(self, target):
        target = target.float()
        with torch.autocast(device_type='cuda'):
            logits, gates = self.backbone(2 * target - 1)
            phase = math.pi * torch.tanh(logits)
        with torch.autocast(device_type='cuda', enabled=False):
            reconstruction = self.physics(phase.float())
            cc, sv = doe_metrics(target, reconstruction)
            metric_loss = (1 - cc).mean() + SSIM_WEIGHT * (1 - sv).mean()
            mean_gate = gates.float().mean(dim=0)
            balance_loss = (mean_gate - 1 / 3).square().sum()
            total_loss = metric_loss + GATE_BALANCE_WEIGHT * balance_loss
        return total_loss.reshape(1), cc, sv, gates


## Кэш, DataParallel и полный train

Первый запуск создаёт около 3.5 GB float16-кэша train/validation. Это одноразовая CPU-стадия; последующие
эпохи не выполняют PIL-кропы и процедурный рендеринг. Общий batch `32` делится на две GPU.

`DataParallel` оборачивает `EndToEndDOE`, поэтому на каждую GPU реплицируются эксперты, Френель и loss.
Сохраняются gate weights и source-wise метрики. Если память позволяет, увеличьте `BATCH_SIZE` до `64`.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Enable GPU acceleration')
DEVICE_IDS = list(range(torch.cuda.device_count()))
if len(DEVICE_IDS) < 2:
    print('WARNING: only one GPU is visible; DataParallel will not be enabled')
primary_device = torch.device(f'cuda:{DEVICE_IDS[0]}')
print('GPUs:', [torch.cuda.get_device_name(i) for i in DEVICE_IDS])

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
set_seed(TRAIN_SEED)

def build_cache(split):
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    image_path, source_path = CACHE_DIR / f'{split}_images.npy', CACHE_DIR / f'{split}_sources.npy'
    done_path = CACHE_DIR / f'{split}.complete'
    manifest = DATA_ROOT / 'manifests' / f'{split}.jsonl'
    base = QuadSourceDataset(manifest)
    if image_path.exists() and source_path.exists() and done_path.exists():
        images = np.load(image_path, mmap_mode='r')
        if images.shape[0] == len(base):
            print(split, 'cache already exists:', image_path)
            return image_path, source_path
    loader = DataLoader(base, batch_size=64, shuffle=False, num_workers=CACHE_WORKERS,
                        pin_memory=False, persistent_workers=CACHE_WORKERS > 0)
    images = np.lib.format.open_memmap(image_path, mode='w+', dtype=np.float16,
                                       shape=(len(base), 4, IMAGE_SIZE, IMAGE_SIZE))
    sources = np.lib.format.open_memmap(source_path, mode='w+', dtype=np.uint8, shape=(len(base), 4))
    offset, started = 0, time.perf_counter()
    for batch_index, (quad, source) in enumerate(loader, start=1):
        size = quad.shape[0]
        images[offset:offset+size] = quad.numpy().astype(np.float16)
        sources[offset:offset+size] = source.numpy().astype(np.uint8)
        offset += size
        if batch_index == 1 or batch_index % 50 == 0 or batch_index == len(loader):
            print(f'cache {split}: {offset}/{len(base)}, {time.perf_counter()-started:.0f}s', flush=True)
    images.flush(); sources.flush(); done_path.write_text('complete', encoding='utf-8')
    return image_path, source_path

class CachedQuadDataset(Dataset):
    def __init__(self, image_path, source_path):
        self.images = np.load(image_path, mmap_mode='r')
        self.sources = np.load(source_path, mmap_mode='r')
    def __len__(self): return self.images.shape[0]
    def __getitem__(self, index):
        image = torch.from_numpy(np.array(self.images[index], copy=True))
        source = torch.from_numpy(np.array(self.sources[index], copy=True)).long()
        return image, source

train_cache = build_cache('train'); val_cache = build_cache('val')
train_dataset = CachedQuadDataset(*train_cache); val_dataset = CachedQuadDataset(*val_cache)

def make_train_loader(epoch):
    generator = torch.Generator().manual_seed(TRAIN_SEED + 10_000 + epoch)
    return DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=generator,
                      num_workers=2, pin_memory=True, persistent_workers=False, prefetch_factor=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
                        pin_memory=True, persistent_workers=True, prefetch_factor=2)

core = EndToEndDOE().to(primary_device)
parameter_count = sum(p.numel() for p in core.backbone.parameters())
system = nn.DataParallel(core, device_ids=DEVICE_IDS) if len(DEVICE_IDS) > 1 else core
optimizer = torch.optim.AdamW(system.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
def lr_multiplier(epoch):
    progress = epoch / max(1, MAX_EPOCHS - 1)
    return (MIN_LR/LR) + (1-MIN_LR/LR) * 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_multiplier)
scaler = torch.amp.GradScaler('cuda')

def unwrap(): return system.module if isinstance(system, nn.DataParallel) else system

if BASE_CHECKPOINT is not None and RESUME_CHECKPOINT is None:
    base = torch.load(BASE_CHECKPOINT, map_location=primary_device, weights_only=False)
    missing, unexpected = unwrap().backbone.load_state_dict(base['model'], strict=False)
    print('Warm start. Missing:', len(missing), 'unexpected:', len(unexpected))

@torch.no_grad()
def validate():
    system.eval(); count = 0
    cc_plane = torch.zeros(4, device=primary_device); ss_plane = torch.zeros(4, device=primary_device)
    gate_sum = torch.zeros(3, device=primary_device)
    source_sums = {0: {'cc':0.,'ssim':0.,'n':0}, 1: {'cc':0.,'ssim':0.,'n':0}}
    for target, sources in val_loader:
        _, cc, sv, gates = system(target)
        batch = cc.shape[0]; count += batch
        cc_plane += cc.sum(0); ss_plane += sv.sum(0); gate_sum += gates.sum(0)
        sources = sources.to(primary_device, non_blocking=True)
        for source_id in (0,1):
            mask = sources == source_id; n = int(mask.sum())
            source_sums[source_id]['cc'] += float(cc[mask].sum())
            source_sums[source_id]['ssim'] += float(sv[mask].sum())
            source_sums[source_id]['n'] += n
    ccv=(cc_plane/count).cpu().tolist(); ssv=(ss_plane/count).cpu().tolist(); gates=(gate_sum/count).cpu().tolist()
    row={'val_loss':float(1-np.mean(ccv)+SSIM_WEIGHT*(1-np.mean(ssv))),
         'val_cc':float(np.mean(ccv)),'val_ssim':float(np.mean(ssv)),
         'val_cc_min':float(np.min(ccv)),'val_ssim_min':float(np.min(ssv)),
         'gate_spatial':gates[0],'gate_transformer':gates[1],'gate_fourier':gates[2]}
    row.update({f'cc_z{i+1}':v for i,v in enumerate(ccv)}); row.update({f'ssim_z{i+1}':v for i,v in enumerate(ssv)})
    for source_id,label in ((0,'synthetic'),(1,'bsds')):
        values=source_sums[source_id]; row[f'cc_{label}']=values['cc']/values['n']; row[f'ssim_{label}']=values['ssim']/values['n']
    return row

def save_checkpoint(path, state):
    temporary=path.with_suffix(path.suffix+'.tmp'); torch.save(state,temporary); temporary.replace(path)
def write_history(history):
    with (OUTPUT_DIR/'metrics.csv').open('w',newline='') as stream:
        writer=csv.DictWriter(stream,fieldnames=history[0].keys()); writer.writeheader(); writer.writerows(history)
    (OUTPUT_DIR/'history.json').write_text(json.dumps(history,indent=2),encoding='utf-8')

history=[]; start_epoch=0; best={'joint_loss':float('inf'),'cc':-float('inf'),'ssim':-float('inf')}; stale=0
if RESUME_CHECKPOINT is not None:
    saved=torch.load(RESUME_CHECKPOINT,map_location=primary_device,weights_only=False)
    unwrap().backbone.load_state_dict(saved['model']); optimizer.load_state_dict(saved['optimizer'])
    scheduler.load_state_dict(saved['scheduler']); scaler.load_state_dict(saved['scaler'])
    start_epoch=saved['epoch']+1; history=saved['history']; best=saved['best']; stale=saved['stale']

for epoch in range(start_epoch,MAX_EPOCHS):
    loader=make_train_loader(epoch); system.train(); started=time.perf_counter(); total=0.; seen=0
    for batch_index,(target,_) in enumerate(loader,start=1):
        optimizer.zero_grad(set_to_none=True)
        loss_vector,_,_,_=system(target); loss=loss_vector.mean()
        scaler.scale(loss).backward(); scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(system.parameters(),1.0); scaler.step(optimizer); scaler.update()
        total += loss.item()*target.shape[0]; seen += target.shape[0]
        if batch_index==1 or batch_index%LOG_EVERY==0 or batch_index==len(loader):
            elapsed=time.perf_counter()-started; eta=elapsed/batch_index*(len(loader)-batch_index)
            print(f'epoch {epoch+1}/{MAX_EPOCHS} batch {batch_index}/{len(loader)} loss={loss.item():.4f} ETA={eta/60:.1f}m',flush=True)
    row=validate(); row.update({'epoch':epoch+1,'train_loss':total/seen,'lr':optimizer.param_groups[0]['lr'],
                               'epoch_seconds':time.perf_counter()-started}); history.append(row); scheduler.step()
    ij=row['val_loss']<best['joint_loss']; ic=row['val_cc']>best['cc']; iss=row['val_ssim']>best['ssim']
    if ij: best['joint_loss']=row['val_loss']; stale=0
    else: stale+=1
    if ic: best['cc']=row['val_cc']
    if iss: best['ssim']=row['val_ssim']
    state={'epoch':epoch,'model':unwrap().backbone.state_dict(),'optimizer':optimizer.state_dict(),
           'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),'history':history,'best':best,
           'stale':stale,'config':CONFIG,'parameter_count':parameter_count,'metrics_at_save':row}
    save_checkpoint(OUTPUT_DIR/'last.pt',state)
    if ij: save_checkpoint(OUTPUT_DIR/'best_joint.pt',state)
    if ic: save_checkpoint(OUTPUT_DIR/'best_cc.pt',state)
    if iss: save_checkpoint(OUTPUT_DIR/'best_ssim.pt',state)
    write_history(history); print(row,flush=True); print('best',best,'stale',stale,flush=True)
    del loader
    if epoch+1>=MIN_EPOCHS and stale>=EARLY_STOP_PATIENCE:
        print('Early stopping',flush=True); break

summary={}
for filename in ('best_joint.pt','best_cc.pt','best_ssim.pt','last.pt'):
    path=OUTPUT_DIR/filename
    if path.exists():
        saved=torch.load(path,map_location='cpu',weights_only=False)
        summary[filename]={'epoch':saved['epoch']+1,'metrics':saved['metrics_at_save']}
(OUTPUT_DIR/'checkpoint_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print(json.dumps(summary,indent=2))

In [ ]:
def phase_to_uint8(phase):
    return ((phase + math.pi) / (2 * math.pi) * 255).round().clamp(0,255).to(torch.uint8)

@torch.no_grad()
def export_validation_example(index=0, checkpoint_path=OUTPUT_DIR/'best_joint.pt'):
    model=HarmonicMoEUNet().to(primary_device)
    saved=torch.load(checkpoint_path,map_location=primary_device,weights_only=False); model.load_state_dict(saved['model']); model.eval()
    target,sources=val_dataset[index]; target=target.float().unsqueeze(0).to(primary_device)
    with torch.autocast(device_type='cuda'):
        logits,gates=model(2*target-1); phase=math.pi*torch.tanh(logits)
    export_dir=OUTPUT_DIR/'validation_export'; export_dir.mkdir(parents=True,exist_ok=True)
    doe=phase_to_uint8(phase[0,0]).cpu().numpy(); Image.fromarray(doe,mode='L').save(export_dir/'doe.png')
    stored=torch.from_numpy(np.asarray(Image.open(export_dir/'doe.png').convert('L'),dtype=np.uint8)).to(primary_device)
    stored_phase=stored.float()[None,None]/255*(2*math.pi)-math.pi
    reconstruction=FresnelLayer(primary_device)(stored_phase).cpu(); target=target.cpu()
    cc=pearson_cc(target,reconstruction); sv=torch.stack([ssim(target[:,m:m+1],reconstruction[:,m:m+1],data_range=1.,size_average=False) for m in range(4)],dim=1)
    for m in range(4):
        Image.fromarray((target[0,m]*255).round().byte().numpy(),mode='L').save(export_dir/f'target_z{m+1}.png')
        Image.fromarray((reconstruction[0,m]*255).round().byte().numpy(),mode='L').save(export_dir/f'reconstruction_z{m+1}.png')
    report={'epoch':saved['epoch']+1,'cc':cc[0].tolist(),'ssim':sv[0].tolist(),'gates':gates[0].cpu().tolist(),
            'sources':['bsds' if int(x)==1 else 'synthetic' for x in sources]}
    (export_dir/'metrics.json').write_text(json.dumps(report,indent=2),encoding='utf-8'); print(json.dumps(report,indent=2))

export_validation_example()